# visualize

> Look at the data and the model: verify image/mask alignment, overlay predictions on inputs, and surface the worst patches for error analysis.

Good segmentation work is mostly looking at pictures. `sanity_check` confirms augmentations keep masks aligned; `show_pred` overlays ground truth vs prediction; `show_top_losses` is the fastest way to find label errors and hard cases.

In [ ]:
#| default_exp visualize

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import math
from pathlib import Path
from typing import List, Optional, Tuple

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch

from segmentation_tools_pt.inference import predict, decode

## Overlay one prediction

Ground-truth-only pixels are green, prediction-only red, agreement magenta. Foreground from all classes is grouped together, so the same view works for binary and multi-class. `pred` may be a label map or a `(C, H, W)` probability map.

In [ ]:
#| export
def _to_numpy(x):
    "Detach a tensor to numpy; pass arrays through."
    return x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)

def show_pred(
    image,                   # (C, H, W) or (H, W, C) input
    mask,                    # (H, W) integer-label ground truth
    pred,                    # (H, W) labels or (C, H, W) probabilities
    threshold: float = 0.5,  # binary decision threshold if pred is probabilities
    ax=None,                 # matplotlib Axes to draw on
):
    """Overlay ground truth vs prediction on the first image channel. Returns the Axes."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    image = _to_numpy(image)
    if image.ndim == 3 and image.shape[0] <= 4:
        image = image.transpose(1, 2, 0)
    mask = _to_numpy(mask).squeeze()
    pred = _to_numpy(pred).squeeze()
    if pred.ndim == 3:
        pred = (pred[0] >= threshold).astype(int) if pred.shape[0] == 1 else pred.argmax(0)
    gt_fg, pr_fg = mask > 0, pred > 0
    overlay = np.zeros((*gt_fg.shape, 3), np.float32)
    overlay[gt_fg & ~pr_fg] = (0.0, 0.8, 0.0)
    overlay[pr_fg & ~gt_fg] = (0.8, 0.0, 0.0)
    overlay[gt_fg & pr_fg] = (1.0, 0.0, 1.0)
    disp = image[..., 0] if image.ndim == 3 else image
    ax.imshow(disp, cmap='gray'); ax.imshow(overlay, alpha=0.5); ax.set_axis_off()
    ax.legend(handles=[mpatches.Patch(color=(0, 0.8, 0), label='GT'),
                       mpatches.Patch(color=(0.8, 0, 0), label='Pred'),
                       mpatches.Patch(color=(1, 0, 1), label='Both')], loc='upper right', fontsize=7)
    return ax

## Headless save helpers

YOLO-style validation JPEGs and fastai-style activation histogram PNGs for training callbacks. Uses the Agg backend so saves work without a display.

In [ ]:
#| export
def _disp_channel(image):
    "First channel of a (C,H,W) or (H,W) image for grayscale display."
    image = _to_numpy(image)
    if image.ndim == 3 and image.shape[0] <= 4:
        image = image.transpose(1, 2, 0)
    return image[..., 0] if image.ndim == 3 else image

def _pred_labels(pred, threshold: float = 0.5):
    pred = _to_numpy(pred).squeeze()
    if pred.ndim == 3:
        return (pred[0] >= threshold).astype(int) if pred.shape[0] == 1 else pred.argmax(0)
    return pred

def overlay_gt_only(image, mask, ax=None):
    """Green GT overlay on the first image channel (YOLO `*_labels.jpg` style)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    disp = _disp_channel(image)
    gt_fg = _to_numpy(mask).squeeze() > 0
    overlay = np.zeros((*gt_fg.shape, 3), np.float32)
    overlay[gt_fg] = (0.0, 0.8, 0.0)
    ax.imshow(disp, cmap='gray'); ax.imshow(overlay, alpha=0.5); ax.set_axis_off()
    return ax

def overlay_pred_only(image, pred, threshold: float = 0.5, ax=None):
    """Red prediction overlay on the first image channel (YOLO `*_pred.jpg` style)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    disp = _disp_channel(image)
    pr_fg = _pred_labels(pred, threshold) > 0
    overlay = np.zeros((*pr_fg.shape, 3), np.float32)
    overlay[pr_fg] = (0.8, 0.0, 0.0)
    ax.imshow(disp, cmap='gray'); ax.imshow(overlay, alpha=0.5); ax.set_axis_off()
    return ax

def save_color_dim_png(hist_stack, path: str | Path, title: str = '') -> None:
    """Save fastai-style colorful-dimension plot: log1p(hist) imshow, bins on y, batches on x."""
    h = _to_numpy(hist_stack).astype(np.float64)
    if h.ndim == 1:
        h = h.reshape(1, -1)
    if h.shape[0] > h.shape[1]:
        h = h.T
    res = np.log1p(h)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(res, origin='lower', aspect='auto', cmap='viridis')
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=9)
    fig.savefig(path, bbox_inches='tight', dpi=100)
    plt.close(fig)

def save_near_zero_png(epochs: List[int], values: List[float], path: str | Path, title: str = '') -> None:
    "Line plot of near-zero activation fraction over epochs."
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(epochs, values, marker='o', markersize=3)
    ax.set_xlabel('epoch'); ax.set_ylabel('% near zero'); ax.set_ylim(0, 1)
    if title:
        ax.set_title(title, fontsize=9)
    fig.savefig(path, bbox_inches='tight', dpi=100)
    plt.close(fig)

def save_overlay_grid_jpeg(
    images,
    masks,
    preds,
    path: str | Path,
    threshold: float = 0.5,
    mode: str = 'compare',
    jpeg_quality: int = 90,
    max_cols: int = 4,
) -> None:
    """Mosaic batch overlays and save as JPEG (`labels`, `pred`, or `compare`)."""
    imgs = _to_numpy(images)
    msks = _to_numpy(masks)
    prds = _to_numpy(preds)
    if imgs.ndim == 3:
        imgs, msks, prds = imgs[None], msks[None], prds[None]
    n = imgs.shape[0]
    rows, cols = _grid(n, min(max_cols, n))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.atleast_1d(axes).flatten()
    for i in range(n):
        img, msk, prd = imgs[i], msks[i], prds[i]
        if mode == 'labels':
            overlay_gt_only(img, msk, ax=axes[i])
        elif mode == 'pred':
            overlay_pred_only(img, prd, threshold, ax=axes[i])
        else:
            show_pred(img, msk, prd, threshold, ax=axes[i])
    for i in range(n, len(axes)):
        axes[i].set_visible(False)
    plt.tight_layout()
    fig.savefig(path, format='jpeg', dpi=100, pil_kwargs={'quality': jpeg_quality}, bbox_inches='tight')
    plt.close(fig)

## Sanity-check the dataset

Before training, plot a few augmented samples to confirm the mask still lines up with the image after geometric transforms.

In [ ]:
#| export
def _grid(n: int, cols: int) -> Tuple[int, int]:
    "Rows/cols for `n` panels."
    return math.ceil(n / cols), cols

def sanity_check(dataset, n: int = 8):
    "Plot `n` (image, mask) pairs to verify alignment."
    n = min(n, len(dataset))
    rows, cols = _grid(n, min(5, n))
    fig, axes = plt.subplots(rows, cols * 2, figsize=(cols * 5, rows * 3))
    axes = np.array(axes).reshape(-1, 2)
    for i in range(n):
        s = dataset[i]
        img = _to_numpy(s['image'])
        img = img[0] if img.ndim == 3 else img
        axes[i, 0].imshow(img, cmap='gray'); axes[i, 0].set_title(str(s['sample_id']), fontsize=7)
        axes[i, 1].imshow(_to_numpy(s['mask']).squeeze(), cmap='gray')
        for a in axes[i]:
            a.set_axis_off()
    for i in range(n, len(axes)):
        for a in axes[i]:
            a.set_visible(False)
    plt.tight_layout()
    return fig

## Predictions and worst cases

In [ ]:
#| export
def show_batch(model, dataset, device, n: int = 8, threshold: float = 0.5):
    "Overlay predictions for `n` random samples."
    idxs = np.random.choice(len(dataset), min(n, len(dataset)), replace=False)
    rows, cols = _grid(len(idxs), min(4, len(idxs)))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).flatten()
    for i, idx in enumerate(idxs):
        s = dataset[int(idx)]
        probs = predict(model, s['image'], device)
        show_pred(s['image'], s['mask'], decode(probs, threshold), threshold, ax=axes[i])
        axes[i].set_title(str(s['sample_id']), fontsize=8)
    for i in range(len(idxs), len(axes)):
        axes[i].set_visible(False)
    plt.tight_layout()
    return fig

@torch.no_grad()
def compute_patch_losses(model, dataset, criterion, device, n: Optional[int] = None) -> List[tuple]:
    "Per-sample loss, sorted worst-first as `(idx, loss, sample_id)`."
    model.eval()
    n = n or len(dataset)
    results = []
    for idx in range(min(n, len(dataset))):
        s = dataset[idx]
        logits = model(s['image'].unsqueeze(0).to(device))
        loss = float(criterion(logits, s['mask'].unsqueeze(0).to(device))['loss'])
        results.append((idx, loss, s['sample_id']))
    results.sort(key=lambda r: r[1], reverse=True)
    return results

def show_top_losses(model, dataset, criterion, device, k: int = 8, threshold: float = 0.5):
    "Overlay the `k` highest-loss samples for error analysis."
    top = compute_patch_losses(model, dataset, criterion, device)[:k]
    rows, cols = _grid(len(top), min(4, len(top)))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).flatten()
    for i, (idx, loss, sid) in enumerate(top):
        s = dataset[idx]
        show_pred(s['image'], s['mask'], decode(predict(model, s['image'], device), threshold), threshold, ax=axes[i])
        axes[i].set_title(f'{sid}\nloss={loss:.4f}', fontsize=7)
    for i in range(len(top), len(axes)):
        axes[i].set_visible(False)
    plt.tight_layout()
    return fig

In [ ]:
# test: overlay, sanity check, headless saves, and worst-first ordering on a tiny dataset
import matplotlib
matplotlib.use('Agg')
import tempfile
import torch.nn as nn
from torch.utils.data import Dataset
from segmentation_tools_pt.losses import SegLoss

class _DS(Dataset):
    def __len__(self): return 4
    def __getitem__(self, i):
        m = torch.zeros(16, 16, dtype=torch.long); m[4:10, 4:10] = 1
        return {'image': torch.rand(3, 16, 16), 'mask': m, 'sample_id': str(i)}

ds = _DS()
assert show_pred(ds[0]['image'], ds[0]['mask'], ds[0]['mask']) is not None
assert sanity_check(ds, n=2) is not None
res = compute_patch_losses(nn.Conv2d(3, 1, 3, padding=1), ds, SegLoss(), torch.device('cpu'))
assert len(res) == 4 and res[0][1] >= res[-1][1]
imgs = torch.stack([ds[i]['image'] for i in range(2)])
msks = torch.stack([ds[i]['mask'] for i in range(2)])
prds = msks.clone()
with tempfile.TemporaryDirectory() as d:
    save_color_dim_png(torch.rand(40, 3), Path(d) / 'color.png')
    save_near_zero_png([0, 1], [0.1, 0.2], Path(d) / 'near_zero.png')
    save_overlay_grid_jpeg(imgs, msks, prds, Path(d) / 'grid.jpg', mode='compare')
    assert (Path(d) / 'color.png').exists()
    assert (Path(d) / 'near_zero.png').exists()
    assert (Path(d) / 'grid.jpg').exists()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()